In [18]:
import pandas as pd
import numpy as np
from typing import List, Literal
from pydantic import BaseModel, Field
from pyspark.sql import functions as F
from pyspark.sql import SparkSession, DataFrame

In [2]:
num_users = int(1e5)
emb_dim = 256
user_emb_table = np.random.randn(num_users, emb_dim)
num_posts = int(1e4)
post_emb_table = np.random.randn(num_posts, emb_dim)

In [3]:
class User(BaseModel):
  user_sk: int = Field(..., ge=0, lt=num_users)
  name: str
  age: int = Field(..., ge=0)
  gender: str = Literal["Male", "Female", "Other"]

class Post(BaseModel):
  post_sk: int = Field(..., ge=0, lt=num_posts)
  title: str
  category: str
  author_sk: int = Field(..., ge=0, lt=num_users)


In [4]:
spark = SparkSession.builder.getOrCreate()

In [5]:
post_df = pd.DataFrame({
    "post_sk": np.arange(num_posts),
    "title": [f"Post Title {i}" for i in range(num_posts)],
    "category": np.random.choice(["Tech", "Sports", "News", "Entertainment"], num_posts),
    "author_sk": np.random.randint(0, num_users, num_posts) # Randomly assign an author from existing users
})
spark_post_df = spark.createDataFrame(post_df)

In [12]:
class MMRConfig(BaseModel):
  lmbda: float = Field(..., ge=0, le=1)
  user: User
  feed_size: int = Field(..., ge=1)

In [31]:
def mmr(mmr_config: MMRConfig,
        spark_post_df: DataFrame,
        user_emb_table: np.array,
        post_emb_table: np.array) -> List[Post]:
  """
  Generates a recommendation feed using Maximal Marginal Relevance (MMR),
  balancing user relevance and post diversity via cosine similarity.

  Args:
      mmr_config: Feed configuration and target user information.
      spark_post_df: Spark DataFrame containing post metadata.
      user_emb_table: User embedding matrix.
      post_emb_table: Post embedding matrix.

  Returns:
      Ordered list of recommended posts.

  Raises:
      ValueError: If feed_size exceeds number of available posts.
  """
  feed_size = mmr_config.feed_size
  num_posts = post_emb_table.shape[0]
  if feed_size > num_posts:
    raise ValueError(f"Feed size {feed_size} will have duplicate"
                     f"posts given {num_posts} posts")

  posts_pdf = spark_post_df.select(
        "post_sk",
        "title",
        "category",
        "author_sk"
  ).toPandas().sort_values("post_sk")

  categories = posts_pdf["category"].values

  ordered_posts = []
  full_indices = set(range(num_posts))
  seen_indices = set()
  seen_categories = set()
  target_sk = mmr_config.user.user_sk
  lmbda = mmr_config.lmbda
  user_emb = user_emb_table[target_sk, :]
  user_norm = np.sqrt(np.sum(user_emb ** 2))
  user_emb /= user_norm
  post_norms = np.linalg.norm(post_emb_table, axis=1, keepdims=True)
  normalized_posts = post_emb_table / np.clip(post_norms, 1e-12, None)
  relevances = normalized_posts @ user_emb

  selected_mask = np.zeros(num_posts, dtype=bool)
  max_redundancy = np.zeros(num_posts)

  for _ in range(feed_size):
    candidate_mask = ~selected_mask
    candidate_indices = np.where(candidate_mask)[0]

    # Topic redundancy vectorized
    topic_redundancy = np.isin(
        categories[candidate_indices],
        list(seen_categories)
    ).astype(np.float32)

    scores = (
        lmbda * relevances[candidate_indices]
        - (1 - lmbda) / 2 * max_redundancy[candidate_indices]
        - (1 - lmbda) / 2 * topic_redundancy
    )

    best_relative_idx = np.argmax(scores)
    best_idx = candidate_indices[best_relative_idx]

    selected_mask[best_idx] = True
    seen_categories.add(categories[best_idx])

    similarities = normalized_posts @ normalized_posts[best_idx]

    max_redundancy = np.maximum(
        max_redundancy,
        similarities
    )
    post_row = posts_pdf.iloc[best_idx]
    ordered_posts.append(Post(post_sk=post_row.post_sk,
               title=post_row.title,
               category=post_row.category,
               author_sk=post_row.author_sk))
  return ordered_posts

mmr_config = MMRConfig(lmbda=0.99,
                       user = User(user_sk=1,
                                   name="User_1",
                                   age=48,
                                   gender="Male"),
                       feed_size=10)

mmr(mmr_config,
    spark_post_df=spark_post_df,
    user_emb_table=user_emb_table,
    post_emb_table=post_emb_table)

[Post(post_sk=4120, title='Post Title 4120', category='News', author_sk=14220),
 Post(post_sk=5413, title='Post Title 5413', category='Tech', author_sk=48636),
 Post(post_sk=542, title='Post Title 542', category='Entertainment', author_sk=21476),
 Post(post_sk=5476, title='Post Title 5476', category='Sports', author_sk=61399),
 Post(post_sk=5036, title='Post Title 5036', category='Tech', author_sk=46185),
 Post(post_sk=6339, title='Post Title 6339', category='Entertainment', author_sk=80638),
 Post(post_sk=5422, title='Post Title 5422', category='Entertainment', author_sk=30816),
 Post(post_sk=1810, title='Post Title 1810', category='Sports', author_sk=43638),
 Post(post_sk=1913, title='Post Title 1913', category='News', author_sk=95051),
 Post(post_sk=6628, title='Post Title 6628', category='Entertainment', author_sk=68888)]